In [1]:
import pandas as pd

In [2]:
from_folder = '/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/data/datasets/stoxx_600/PDFs/'
to_folder = '/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/data/datasets/stoxx_600/company_descriptions/'
df_overview_path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/data/datasets/stoxx_600/stoxx_600_overview.csv"

In [3]:
df_overview = pd.read_csv(df_overview_path, sep=";")
df_overview = df_overview.dropna(subset="description_page")

In [4]:
df_overview

,Unnamed: 0,Name,Symbol,FactSet ID,Revenue - 2022 (in EUR),Revenue - 2023 (in EUR),Revenue - 2024 (in EUR),NACE,NACE_letter,Report,description_page
3,107,AAK AB,AAK-SE,AAK-SE,4741.086317,4010.433824,3939.34280627966,10.89,C,AAK AB1.pdf,3
6,94,ABB Ltd.,ABBN-CH,ABBN-CH,28073.948473,29665.651960,30586.3762461154,27.11,C,ABB Ltd.2.pdf,18
9,135,Accelleron Industries AG,ACLN-CH,ACLN-CH,742.680345,846.217532,NaN,28.11,C,Accelleron Industries AG1.pdf,7
10,296,Acciona SA,ANA-ES,ANA-ES,11195.000000,17021.000000,19190,41.20,F,Acciona SA2.pdf,7
11,365,Accor SA,AC-FR,AC-FR,4224.000000,5056.000000,5606,55.10,I,Accor SA1.pdf,5
...,...,...,...,...,...,...,...,...,...,...,...
394,69,Orkla ASA,ORK-NO,ORK-NO,5774.960695,5932.590483,6073.67720031738,10.89,C,Orkla ASA1.pdf,11
466,598,Scout24 SE,G24-DE,G24-DE,447.539000,509.114000,s,96.09,S,Scout24 SE3.pdf,41
472,286,Severn Trent Plc,SVT-GB,SVT-GB,2505.264229,2709.430915,NaN,36.00,E,Severn Trent Plc1.pdf,8
479,133,Siemens Energy AG,ENR-DE,ENR-DE,29005.000000,31119.000000,34465,27.33,C,Siemens Energy AG1.pdf,5


In [ ]:
import os
from typing import Iterable, Tuple, Optional
from PyPDF2 import PdfReader, PdfWriter

def copy_pdf_pages(src_path: str,
                   dest_path: str,
                   pages: Optional[Iterable[int]] = None,
                   page_range: Optional[Tuple[int, int]] = None,
                   overwrite: bool = False) -> str:
    """
    Copy a subset of pages from a PDF to a new PDF.

    Parameters:
    - src_path: path to source PDF.
    - dest_path: path for resulting PDF (directories will be created if needed).
    - pages: iterable of 1-based page numbers to copy (e.g. [1,3,5]). Can contain floats (will be cast to int).
             If provided, this is used.
    - page_range: tuple (start, end) 1-based inclusive page range to copy (e.g. (2,5)).
                  Used only if `pages` is None.
    - overwrite: if True, overwrite dest_path if it exists; otherwise raises FileExistsError.

    Returns:
    - dest_path on success.

    Raises:
    - FileNotFoundError if src_path does not exist.
    - ValueError for invalid page numbers or ranges.
    - FileExistsError if dest exists and overwrite is False.
    """
    if not os.path.isfile(src_path):
        raise FileNotFoundError(f"Source PDF not found: {src_path}")

    dest_dir = os.path.dirname(dest_path) or "."
    os.makedirs(dest_dir, exist_ok=True)

    if os.path.exists(dest_path) and not overwrite:
        raise FileExistsError(f"Destination already exists: {dest_path}")

    reader = PdfReader(src_path)
    num_pages = len(reader.pages)

    # determine zero-based page indices to copy
    pages_to_copy = []
    if pages is not None:
        for p in pages:
            if p is None:
                continue
            try:
                p_int = int(p)
            except Exception:
                raise ValueError(f"Invalid page value: {p}")
            if not (1 <= p_int <= num_pages):
                raise ValueError(f"Page number out of range: {p_int} (valid 1..{num_pages})")
            pages_to_copy.append(p_int - 1)
    elif page_range is not None:
        start, end = page_range
        try:
            start_i = int(start)
            end_i = int(end)
        except Exception:
            raise ValueError(f"Invalid page_range values: {page_range}")
        if not (1 <= start_i <= end_i <= num_pages):
            raise ValueError(f"Invalid page_range: {(start_i, end_i)} (valid 1..{num_pages})")
        pages_to_copy = list(range(start_i - 1, end_i))
    else:
        # copy all pages
        pages_to_copy = list(range(num_pages))

    writer = PdfWriter()
    for idx in pages_to_copy:
        writer.add_page(reader.pages[idx])

    # write output
    
    with open(dest_path, "wb") as f_out:
        writer.write(f_out)

    return dest_path

In [6]:
import ast
import numpy as np

df_overview["description_page"]
list([ast.literal_eval(df_overview["description_page"].iloc[3])])
np.array([ast.literal_eval(df_overview["description_page"].loc[26])]).flatten()

array([30, 31])

In [7]:
for i, row in df_overview.iterrows():
    copy_pdf_pages(from_folder + row["Report"], to_folder + row["Report"], np.array([ast.literal_eval(row["description_page"])]).flatten(), overwrite=True)    